In [16]:
import pygame, random, sys

# Written and tested by me
# Structure and game loop logic assisted by ChatGPT

pygame.init()
W, H = 800, 300
screen = pygame.display.set_mode((W, H))
pygame.display.set_caption("Endless Runner")
clock = pygame.time.Clock()

# --- Constants ---
GROUND     = 240
GRAVITY    = 0.8
JUMP_VY    = -15
BASE_SPEED = 6

STAND_H = 50
DUCK_H  = 25

# --- Colours ---
SKY      = (135, 206, 235)
GRASS    = (80,  160,  80)
PLAYER_C = (70,  130, 200)
DUCK_C   = (50,  100, 170)   
OBS_C    = (60,  150,  60)
OBS2_C   = (0,     0,   0)
HEART_C  = (220,  50,  50)
WHITE    = (255, 255, 255)
BLACK    = (0,    0,   0)
RED      = (200,  0,   0)
DARK     = (30,   30,  30)

font_big = pygame.font.SysFont(None, 64)
font_med = pygame.font.SysFont(None, 36)
font_sm  = pygame.font.SysFont(None, 26)

# --- Game state ---
# States: "menu", "playing", "dead"
state      = "menu"
high_score = 0   # session only

def reset_game():
    return {
        "player":      pygame.Rect(100, GROUND - STAND_H, 40, STAND_H),
        "vy":          0,
        "on_ground":   True,
        "ducking":     False,
        "obstacles":   [],
        "score":  0,
        "speed":  BASE_SPEED,
        "spawn":  0,
        "lives":       3,
        "invincible":  0,
    }

g = reset_game()

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def spawn_obstacle():
    """Return a new obstacle rect.
    Type 0 - tall ground cactus (jump over)
    Type 1 - wide low log      (jump over)
    Type 2 - floating bird     (duck under)
    """
    kind = random.randint(0, 2)
    if kind == 0:
        h = random.randint(40, 70)
        return pygame.Rect(W, GROUND - h, 25, h), kind
    elif kind == 1:
        h = random.randint(20, 35)
        return pygame.Rect(W, GROUND - h, 55, h), kind
    else:
        h   = 22
        y_max = GROUND - DUCK_H - h - 1   
        y_min = y_max - 20                 
        y = random.randint(y_min, y_max)
        return pygame.Rect(W, y, 40, h), kind

def draw_hearts(lives):
    for i in range(3):
        colour = HEART_C if i < lives else (100, 100, 100)
        x = W - 30 - i * 28
        pygame.draw.polygon(screen, colour, [
            (x,      22),
            (x - 9,  12), (x - 9,   6),
            (x,       9),
            (x + 9,   6), (x + 9,  12),
        ])

# ------------------------------------------------------------------
# Screens
# ------------------------------------------------------------------

def draw_menu():
    screen.fill(SKY)
    pygame.draw.rect(screen, GRASS, (0, GROUND, W, H - GROUND))
    title = font_big.render("ENDLESS RUNNER", True, DARK)
    screen.blit(title, (W // 2 - title.get_width() // 2, 60))
    screen.blit(font_med.render("SPACE / UP  -  Jump", True, BLACK),
                (W // 2 - 130, 145))
    screen.blit(font_med.render("DOWN / SHIFT  -  Duck", True, BLACK),
                (W // 2 - 145, 178))
    screen.blit(font_med.render("SPACE  -  Start    Q  -  Quit", True, BLACK),
                (W // 2 - 200, 220))
    if high_score > 0:
        hs = font_sm.render(f"Session best: {high_score}", True, DARK)
        screen.blit(hs, (W // 2 - hs.get_width() // 2, 260))

def draw_game_over(score, high_score):
    overlay = pygame.Surface((W, H), pygame.SRCALPHA)
    overlay.fill((0, 0, 0, 120))
    screen.blit(overlay, (0, 0))
    go = font_big.render("GAME OVER", True, RED)
    screen.blit(go, (W // 2 - go.get_width() // 2, 70))
    sc = font_med.render(f"Score: {score}",      True, WHITE)
    hs = font_med.render(f"Best:  {high_score}", True, WHITE)
    screen.blit(sc, (W // 2 - sc.get_width() // 2, 148))
    screen.blit(hs, (W // 2 - hs.get_width() // 2, 182))
    screen.blit(font_med.render("SPACE - Restart     Q - Quit", True, WHITE),
                (W // 2 - 195, 230))

# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------

running = True

while running:
    clock.tick(60)

    # ---- Events ----
    for e in pygame.event.get():
        if e.type == pygame.QUIT:
            running = False

        if e.type == pygame.KEYDOWN:
            if e.key == pygame.K_q:
                running = False

            if e.key in (pygame.K_SPACE, pygame.K_UP):
                if state == "menu":
                    g = reset_game()
                    state = "playing"
                elif state == "dead":
                    g = reset_game()
                    state = "playing"
                elif state == "playing" and g["on_ground"] and not g["ducking"]:
                    g["vy"] = JUMP_VY
                    g["on_ground"] = False

    # ---- Duck ----
    if state == "playing":
        keys = pygame.key.get_pressed()
        ducking = keys[pygame.K_DOWN] or keys[pygame.K_LSHIFT] or keys[pygame.K_RSHIFT]

        if ducking and not g["ducking"] and g["on_ground"]:
            g["player"].height = DUCK_H
            g["player"].y      = GROUND - DUCK_H
            g["ducking"]       = True
        elif not ducking and g["ducking"]:
            g["player"].height = STAND_H
            g["player"].y      = GROUND - STAND_H
            g["ducking"]       = False

    # ---- Update ----
    if state == "playing":
        # Physics
        if not g["ducking"]:
            g["vy"] += GRAVITY
            g["player"].y += int(g["vy"])
            if g["player"].y >= GROUND - STAND_H:
                g["player"].y  = GROUND - STAND_H
                g["vy"]        = 0
                g["on_ground"] = True

        # Spawn
        g["spawn"] += 1
        if g["spawn"] > random.randint(40, 75):
            g["obstacles"].append(spawn_obstacle())
            g["spawn"] = 0

        # Move obstacles
        for obs, kind in g["obstacles"]:
            obs.x -= g["speed"]
        g["obstacles"] = [(o, k) for o, k in g["obstacles"] if o.right > 0]

        # Score & speed
        g["score"] += 1
        g["speed"]  = BASE_SPEED + g["score"] // 250

        # Invincibility countdown
        if g["invincible"] > 0:
            g["invincible"] -= 1

        # Collision
        if g["invincible"] == 0:
            for obs, kind in g["obstacles"]:
                if kind == 2 and g["ducking"]:
                    continue   
                if g["player"].colliderect(obs):
                    g["lives"] -= 1
                    if g["lives"] > 0:
                        g["score"] = int(g["score"] * 0.75)
                        g["speed"] = BASE_SPEED + g["score"] // 180
                    g["invincible"] = 120
                    g["player"].y   = GROUND - g["player"].height
                    g["vy"]         = 0
                    g["on_ground"]  = True
                    if g["lives"] <= 0:
                        high_score = max(high_score, g["score"])
                        state = "dead"
                    break

    # ---- Draw ----
    screen.fill(SKY)
    pygame.draw.rect(screen, GRASS, (0, GROUND, W, H - GROUND))

    if state == "menu":
        draw_menu()

    elif state in ("playing", "dead"):
        # Player flashes when invincible, darker when ducking
        p_colour = DUCK_C if g["ducking"] else PLAYER_C
        if state == "playing":
            if g["invincible"] == 0 or (g["invincible"] // 6) % 2 == 0:
                pygame.draw.rect(screen, p_colour, g["player"])
        else:
            pygame.draw.rect(screen, p_colour, g["player"])

        for obs, kind in g["obstacles"]:
            pygame.draw.rect(screen, OBS2_C if kind == 2 else OBS_C, obs)

        screen.blit(font_med.render(f"Score: {g['score']}",  True, BLACK), (10, 10))
        screen.blit(font_sm .render(f"Best:  {high_score}",  True, DARK),  (10, 42))
        draw_hearts(g["lives"])

        if state == "dead":
            draw_game_over(g["score"], high_score)

    pygame.display.flip()

pygame.quit()
sys.exit()

SystemExit: 